# Phase 2 reproduction: Colab smoke run

First end-to-end pass on Colab: clone the repo, install deps, confirm the GPU, and
run the pair **eligibility check** (the one part of `reproduce.py` that runs without
the `kvtransfer`/`lm-eval` integration wired up).

**Before running:** Runtime > Change runtime type > GPU. Use an **L4 (Pro)** or
**A100 (Pro+)** for the real reproduction; a free **T4** is fine for this smoke run.
See `docs/gpu-setup.md`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. (Optional) mount Drive for checkpoints, so a disconnect does not lose the
#    HF cache, the fitted mapper, or results. Skip for the eligibility smoke run.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.environ['HF_HOME'] = '/content/drive/MyDrive/kv-transfer-safety/hf_cache'
    print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# 3. Clone the (private) repo. Colab will prompt for GitHub auth on a private clone;
#    use a Personal Access Token as the password, or upload/copy the repo instead.
REPO_URL = 'https://github.com/TELLON2001/kv-transfer-safety.git'
import os
if not os.path.isdir('kv-transfer-safety'):
    !git clone $REPO_URL
%cd kv-transfer-safety
!git log --oneline -1

**No GitHub remote yet?** Either zip the local repo and use the file-upload button,
or keep it on Drive and `!cp -r /content/drive/MyDrive/kv-transfer-safety .` , then
`%cd kv-transfer-safety`.

In [ ]:
# 4. Install dependencies. torch is preinstalled on Colab; the rest come from the repo.
!pip install -q -r requirements.txt
# The mapper implementation is not on PyPI; pin a commit once you know it:
# !pip install -q "git+https://github.com/Susmith4710/kvtransfer.git@<commit-sha>"

In [ ]:
# 5. Confirm CUDA + bf16 support (bf16 matters for a faithful reproduction).
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
# 6. Smoke test: run the eligibility check for the pair. This downloads only the
#    model configs + tokenizers (small), not the weights, and confirms the pair is
#    mappable (shared tokenizer, matched KV heads / head_dim).
from src.config import load_config
from src.eligibility import check_pair

cfg = load_config('configs/qwen3-1p7b-to-4b.yaml')
print(cfg['pair_id'], ':', cfg['sharer']['model_id'], '->', cfg['receiver']['model_id'])
report = check_pair(cfg)
print('checks:', report.checks)
print('OK' if report.ok else ('PROBLEMS: ' + '; '.join(report.problems)))

## Next steps

If eligibility passes, fill in before the real reproduction:

1. The `null` fields in `configs/qwen3-1p7b-to-4b.yaml`: `num_key_value_heads` and
   `head_dim` (from the printed checks), and `reproduction.task` +
   `reference_accuracy` from the paper.
2. The three `INTEGRATE` points in `scripts/reproduce.py` (mapper build/load, mapped-
   cache receiver, lm-eval call), against the real `kvtransfer` API.
3. Then: `!python scripts/reproduce.py --config configs/qwen3-1p7b-to-4b.yaml`